# 06 – Modellinterpretation, kritische Diskussion & Ethik

## Ziel dieses Notebooks

Dieses Notebook beantwortet zwei zentrale Teilfragen des Projekts:

- **Teilfrage 1:** Welche Merkmale sind für die Vorhersage des Kreditausfalls am wichtigsten, und ist das fachlich plausibel?
- **Teilfrage 5:** Welche ethischen, methodischen und rechtlichen Grenzen hat das Modell?

## Aufbau

| Abschnitt | Methode | Zweck |
|-----------|---------|-------|
| 1. Permutation Importance | Modell-agnostisch, auf dem Testset | *Wie wichtig* ist jedes Merkmal? |
| 2. Impurity-based Importance | Nur Tree-Modelle, Trainingsdaten | Ergänzender Vergleich |
| 3. Partial Dependence Plots | Modell-agnostisch | *Wie* wirkt jedes Merkmal (Richtung & Form)? |
| 4. Plausibilitätsprüfung | Fachlich | Sind die Ergebnisse inhaltlich sinnvoll? |
| 5. Kritische Diskussion & Ethik | Konzeptuell | Grenzen, Bias, Rechtsrahmen |
| 6. Limitationen | Konzeptuell | Ehrliche Einschränkungen dieses Projekts |

> **Grundsatz:** Alle hier betrachteten Interpretations-Maße sind *assoziativ*, nicht kausal.  
> „Wichtig für das Modell" bedeutet **nicht** „Ursache des Ausfalls".

In [6]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, utils, preprocessing as prep, train_models as tm, interpretation as itp
feat = utils.load_processed("feature_matrix")
X_train, X_test, y_train, y_test = prep.make_train_test_split(feat)
fitted = tm.fit_all_models(X_train, y_train)

15:55:44 | INFO    | Split: Train=246008 (Positivrate 0.081), Test=61503 (Positivrate 0.081)
15:55:44 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.
15:55:52 | INFO    | Gefittet: dummy
15:56:20 | INFO    | Gefittet: logreg
15:56:55 | INFO    | Gefittet: random_forest
c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^

## 1. Permutation Importance (modellagnostisch)

### Was ist Permutation Importance?

Für jedes Merkmal wird gemessen, wie stark die Modellleistung (hier: Average Precision / AUC-PR) sinkt, wenn die Werte dieses Merkmals **zufällig permutiert** werden – d. h. die Verbindung zwischen dem Merkmal und der Zielvariable künstlich zerstört wird.

- **Großer Abfall** → Merkmal ist wichtig; das Modell stützt sich stark darauf.
- **Kein oder geringer Abfall** → Merkmal trägt kaum zur Vorhersage bei.

### Warum diese Methode bevorzugt?

| Eigenschaft | Permutation Importance | Impurity-based (→ Abschn. 2) |
|-------------|------------------------|------------------------------|
| Modell-agnostisch | ✓ (jedes Modell) | ✗ (nur Tree-Modelle) |
| Auf Testdaten auswertbar | ✓ | ✗ (Trainingsdaten) |
| Verzerrung bei hochkardinalen Merkmalen | gering | hoch |
| Rechenaufwand | mittel (`n_repeats × Inference`) | sehr gering |

### Interpretation der Werte

- `importance_mean`: durchschnittlicher AUC-PR-Abfall über alle Wiederholungen.
- `importance_std`: Streuung über die Wiederholungen. Hohe Streuung = instabile Wichtigkeit (z. B. bei korrelierten Merkmalen).

### Wichtige Einschränkung: korrelierte Merkmale

Bei stark korrelierten Merkmalen (z. B. `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`) kann die Wichtigkeit auf mehrere Merkmale „aufgeteilt" werden: Nach der Permutation von Merkmal A kann das Modell noch immer auf das korrelierte Merkmal B zurückgreifen und den Schaden begrenzen. Die Einzelwerte **unterschätzen** damit die gemeinsame Bedeutung der gesamten Merkmalsgruppe.

In [4]:
model = fitted["random_forest"]
pi = itp.permutation_importances(model, X_test, y_test, n_repeats=10)
plt.show()
pi.head(15).round(4)

13:45:16 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\interpretation\permutation_importance.csv
13:45:21 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\interpretation\permutation_importance.png
13:45:21 | INFO    | Permutation Importance berechnet. Top-3: ['external_score_mean', 'external_score_min', 'external_score_max']
C:\Users\annis\AppData\Local\Temp\ipykernel_14956\2969259280.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,importance_mean,importance_std
0,external_score_mean,0.0246,0.0021
1,external_score_min,0.0177,0.0019
2,external_score_max,0.0095,0.0011
3,EXT_SOURCE_3,0.0087,0.0008
4,EXT_SOURCE_2,0.0085,0.0008
5,annuity_credit_ratio,0.0038,0.0009
6,NAME_EDUCATION_TYPE,0.0031,0.0006
7,goods_credit_ratio,0.0030,0.0009
8,age_years,0.0030,0.0006
9,AMT_ANNUITY,0.0024,0.0008


### Interpretation der Ergebnisse

Die Top-5 Merkmale sind ausschließlich **externe Bonitätsscores** – das ist das wichtigste Einzelergebnis des Projekts.

- **`external_score_mean` (Δ AUC-PR ≈ 0.025, Rang 1):** Der gemittelte Score aus mehreren Auskunfteien ist mit Abstand der stärkste Prädiktor. Entfernen dieses Merkmals kostet das Modell etwa 25 % seiner Diskriminierungsfähigkeit (relativ zur Baseline-AP). Das unterstreicht, wie sehr das Modell auf extern aggregiertes Wissen angewiesen ist.

- **`external_score_min` und `_max` (Ränge 2–3):** Die Spannbreite der Einzelscores (niedrigster vs. höchster) liefert zusätzliche Information über die Konsistenz der externen Einschätzung. Ein geringer `_min`-Wert signalisiert zumindest eine Auskunftei mit negativer Beurteilung – ein starkes Warnsignal.

- **`EXT_SOURCE_2` und `EXT_SOURCE_3` (Ränge 4–5):** Die Einzelscores bestätigen den Befund; sie werden aber schwächer gewichtet als der aggregierte Mittelwert, weil dieser die Streuung bereits teilweise abbildet.

- **`annuity_credit_ratio` (Rang 6, Δ ≈ 0.004):** Verhältnis monatliche Rate / Kreditbetrag – misst direkt die Schuldentragfähigkeit. Etwa sechsmal schwächer als der externe Score, aber der stärkste intern berechnete Prädiktor.

- **`INST_LATE_PAYMENT_RATIO` (Rang 12):** Der Anteil verspäteter Ratenzahlungen aus früheren Home-Credit-Krediten liefert direktes Verhaltenswissen – trotz der zeitlichen Aggregation noch informativ.

**Kernbotschaft:** Das Modell „delegiert" seinen wichtigsten Informationsgewinn an externe Auskunfteien. Wären diese Scores nicht verfügbar, würde die Modellleistung erheblich sinken.

In [5]:
print(itp.plausibility_check(pi))

Top-5 Merkmale: ['external_score_mean', 'external_score_min', 'external_score_max', 'EXT_SOURCE_3', 'EXT_SOURCE_2']. Keine als sensibel markierten Merkmale unter den Top-5.


## 2. Impurity-based Importance (ergänzend – mit Vorbehalt)

### Was ist Impurity-based Importance?

Bei Entscheidungsbaum-Ensembles (Random Forest, Gradient Boosting) wird bei **jedem Split** aufgezeichnet, wie stark die Unreinheit (Gini-Index bzw. Entropie) durch das jeweilige Merkmal reduziert wird. Die Summe dieser Reduktionen über alle Bäume ergibt die Impurity-based Importance – schnell und ohne zusätzliche Berechnungen.

### Warum nur ergänzend?

Diese Methode hat bekannte **systematische Verzerrungen**:

1. **Kardinalitäts-Bias:** Kontinuierliche Merkmale mit vielen einzigartigen Werten (z. B. Beträge in €) können an mehr Stellen gesplittet werden und werden dadurch systematisch höher bewertet als kategoriale Merkmale mit wenigen Ausprägungen – unabhängig von der tatsächlichen Prädiktionskraft.
2. **Training-Set-Artefakt:** Die Importance wird auf Trainingsdaten berechnet. Überangepasste Merkmale erhalten dadurch überhöhte Werte.
3. **Schwer interpretierbare Absolutwerte:** Erst der Vergleich zwischen Merkmalen ist aussagekräftig.

### Wann trotzdem nützlich?

Als **schnelles Screening** vor der aufwändigeren Permutation Importance. Wenn beide Methoden dieselben Top-Merkmale zeigen, steigt die Konfidenz in die Rangordnung. Wo sie abweichen, lohnt eine genauere Analyse.

### Vergleich: Permutation Importance vs. Impurity-based Importance

| Rang | Permutation Importance | Impurity-based Importance |
|------|------------------------|--------------------------|
| 1 | `external_score_mean` | `external_score_mean` |
| 2 | `external_score_min` | `external_score_min` |
| 3 | `external_score_max` | `external_score_max` |
| 4 | `EXT_SOURCE_3` | `EXT_SOURCE_2` |
| 5 | `EXT_SOURCE_2` | `EXT_SOURCE_3` |
| 6 | `annuity_credit_ratio` | `annuity_credit_ratio` |
| ~7 | `NAME_EDUCATION_TYPE` | **`DAYS_EMPLOYED`** |

**Hohe Übereinstimmung in den oberen Rängen:** Beide Methoden zeigen dieselben Top-6-Merkmale (in leicht unterschiedlicher Reihenfolge bei den EXT_SOURCE-Scores). Das gibt **Konfidenz** in die Rangordnung.

**Abweichung bei `DAYS_EMPLOYED`:**  
`DAYS_EMPLOYED` erscheint in der Impurity-Importance auf Rang ~7, in der Permutation Importance aber deutlich weiter hinten. Mögliche Erklärungen:
- `DAYS_EMPLOYED` und `age_years` sind stark korreliert (ältere Personen → längere Beschäftigungszeit). Bei der Permutation wird die Wichtigkeit zwischen diesen Merkmalen aufgeteilt – jedes erscheint schwächer.
- Die Impurity-Importance bevorzugt kontinuierliche Merkmale mit vielen Splits (Kardinalitäts-Bias).

**Fazit:** Die Übereinstimmung in den Top-5 gibt Vertrauen. Für konkrete Entscheidungen (z. B. Merkmal entfernen) ist die Permutation Importance als robustere Methode ausschlaggebend.

In [6]:
ti = itp.tree_feature_importance(model, top=15)
ti.head(15).round(4) if ti is not None else "Modell ohne impurity-based Importance." 

15:16:39 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\interpretation\tree_feature_importance.csv


,feature,importance
0,num__external_score_mean,0.0475
1,num__external_score_min,0.0391
2,num__external_score_max,0.0343
3,num__EXT_SOURCE_2,0.0292
4,num__EXT_SOURCE_3,0.0271
5,num__annuity_credit_ratio,0.0173
6,num__DAYS_EMPLOYED,0.0171
7,num__external_score_std,0.0168
8,num__DAYS_BIRTH,0.0167
9,num__age_years,0.0161


## 3. Partial Dependence Plots (PDP) – Richtung & Form des Einflusses

### Was zeigen Partial Dependence Plots?

Ein PDP zeigt die **marginale Wirkung** eines einzelnen Merkmals auf die vorhergesagte Ausfallwahrscheinlichkeit – gemittelt über alle anderen Merkmale. Während Permutation Importance beantwortet *wie wichtig* ein Merkmal ist, beantwortet der PDP **wie** es wirkt:

| Verlauf des PDP | Bedeutung |
|-----------------|-----------|
| **Monoton fallend** | Höherer Wert → niedrigere Ausfallwahrscheinlichkeit (Schutzmerkmal) |
| **Monoton steigend** | Höherer Wert → höhere Ausfallwahrscheinlichkeit (Risikofaktor) |
| **Nicht-linear / U-förmig** | Komplexer Zusammenhang – wäre mit einem linearen Modell nicht abbildbar |

**Erwartung für externe Bonitätsscores:** Ein höherer Score (bessere externe Bonität) sollte die vorhergesagte Ausfallwahrscheinlichkeit monoton senken. Wenn das PDP diese Erwartung bestätigt, gibt das fachliche Plausibilität.

### Einschränkungen von PDPs

- **Interaktionen werden ignoriert:** Der Plot mittelt über alle Kombinationen anderer Merkmale. Bei starken Interaktionen (z. B. Alter × Einkommen) kann ein PDP irreführend sein.
- **Korrelierte Merkmale:** Wenn permutierte Werte außerhalb des tatsächlichen gemeinsamen Wertebereichs mehrerer Merkmale liegen, entstehen unrealistische Datenpunkte.
- **Keine Kausalität:** Auch PDPs zeigen assoziative, keine kausalen Zusammenhänge.

In [9]:
from sklearn.inspection import PartialDependenceDisplay

# Sicherheitsnetz: model und pi laden, falls Zelle unabhängig ausgeführt wird
if "model" not in dir():
    model = fitted["random_forest"]
if "pi" not in dir():
    pi = itp.permutation_importances(model, X_test, y_test, n_repeats=5, save=False)

# Top-3 Merkmale aus Permutation Importance
top_features = pi.head(3)["feature"].tolist()
print(f"PDP für Top-3 Merkmale: {top_features}")
print("(Berechnung auf 1000 Testinstanzen – dauert einige Sekunden ...)")

# Nur float32-Spalten → float64 (kategoriale Spalten bleiben unverändert)
X_pdp = X_test.iloc[:1000].copy()
float32_cols = X_pdp.select_dtypes(include="float32").columns
X_pdp[float32_cols] = X_pdp[float32_cols].astype("float64")

fig, axes = plt.subplots(1, len(top_features), figsize=(14, 4))

for i, feat_name in enumerate(top_features):
    PartialDependenceDisplay.from_estimator(
        model, X_pdp, [feat_name],
        ax=axes[i], kind="average"
    )
    axes[i].set_title(f"PDP: {feat_name}", fontsize=10)
    axes[i].set_ylabel("Ø P(Ausfall)", fontsize=9)
    axes[i].set_xlabel(feat_name, fontsize=9)

fig.suptitle("Partial Dependence Plots – Top-3 Merkmale (Random Forest)",
             fontsize=11, y=1.02)
fig.tight_layout()
utils.save_figure(fig, "partial_dependence_top3", subdir="interpretation")
plt.show()
print("Abbildung gespeichert.")

PDP für Top-3 Merkmale: ['external_score_mean', 'external_score_min', 'external_score_max']
(Berechnung auf 1000 Testinstanzen – dauert einige Sekunden ...)


17:03:30 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\interpretation\partial_dependence_top3.png


Abbildung gespeichert.


C:\Users\annis\AppData\Local\Temp\ipykernel_18096\2287146151.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Kurzinterpretation der PDP-Grafik

Die Grafik zeigt die **Partial Dependence Plots (PDP)** für die drei wichtigsten Modellmerkmale (externe Bonitätsscores).

* **Zusammenhang (Verlauf):** Alle Kurven sind **monoton fallend**. Das heißt: Je höher der externe Score (x-Achse), desto niedriger ist die vom Modell vorhergesagte Ausfallwahrscheinlichkeit (y-Achse).
* **Fachliche Plausibilität:** Das Modell verhält sich exakt wie erwartet. Die Scores fungieren als **Schutzmerkmale** – gute Bonität senkt das Risiko (von ca. 10 % auf ca. 6 %).
* **Datenverteilung (Rug-Plots):** Die kleinen Striche an der x-Achse markieren die Dichte der realen Daten. Wo viele Striche sind, ist die Schätzung des Modells am verlässlichsten (meist im Bereich zwischen 0.4 und 0.8).
---

## 4. Fachliche Plausibilitätsprüfung

### Sind die wichtigen Merkmale inhaltlich sinnvoll?

**Externe Bonitätsscores (Plätze 1–5):**  
`external_score_mean`, `_min`, `_max`, `EXT_SOURCE_2`, `EXT_SOURCE_3` sind Scores aus externen Auskunfteien und repräsentieren eine aggregierte Einschätzung der bisherigen Kreditwürdigkeit. Dass diese Merkmale dominieren, ist **hochgradig plausibel** – externe Kreditscores sind auch in der Bankpraxis das wichtigste Instrument der Bonitätsbewertung. Gleichzeitig zeigt es: Das Modell reproduziert im Wesentlichen, was Auskunfteien bereits wissen.

**Verschuldungskennzahlen (`annuity_credit_ratio`, `goods_credit_ratio`):**  
Die monatliche Rate im Verhältnis zur Kreditsumme bzw. zum Warenwert misst direkt die **finanzielle Tragbarkeit**. Hohe Quoten signalisieren Überdehnung – plausibel.

**Zahlungsverhalten (`INST_LATE_PAYMENT_RATIO`, `PREV_APPROVAL_RATE`):**  
Verspätete Raten aus früheren Krediten und eine niedrige Bewilligungsquote bei früheren Anträgen sind klassische Risikoindikatoren – konsistent mit dem akademischen Schrifttum zum Credit Scoring (Thomas, 2009).

**`age_years` und `NAME_EDUCATION_TYPE`:**  
Diese Merkmale haben statistischen Vorhersagewert. Sie sind jedoch ethisch heikel: Alter ist ein geschütztes Merkmal (AGG), Bildungsabschluss kann als Proxy für Einkommensgruppe oder Migrationsgeschichte wirken. Ihr Auftreten in den Top-10 ist zu dokumentieren und im Ethikteil zu diskutieren.

**Kein Warnsignal:**  
IDs, Proxy-Merkmale wie Wohnregion oder offensichtliche Datenlecks dominieren die Top-Ränge nicht. Das ist positiv – kein unmittelbares Leakage, kein offen diskriminierendes Merkmal in den Top-5.

## 5. Kritische Diskussion & Ethik

### 5.1 Datenqualität & fehlende Werte

Viele Merkmale fehlen in erheblichem Umfang. Der Fehlmechanismus ist unbekannt – es gibt drei Fälle:

| Mechanismus | Beschreibung | Problem |
|-------------|-------------|---------|
| **MCAR** (Missing Completely At Random) | Zufälliges Fehlen, unabhängig von allen Variablen | Imputation verzerrt kaum |
| **MAR** (Missing At Random) | Fehlen hängt von *anderen beobachteten* Merkmalen ab | Bedingt unproblematisch |
| **MNAR** (Missing Not At Random) | Fehlen hängt vom Wert selbst ab (z. B. Einkommen fehlt häufiger bei Niedrigeinkommen) | Imputation kann systematisch verzerren und Diskriminierung verstärken |

Die verwendete Median/Mode-Imputation setzt MCAR voraus – wahrscheinlich eine Vereinfachung. Robustere Alternativen: Multiple Imputation (MICE), modellbasierte Imputation.

### 5.2 Historische Verzerrungen (Feedback-Bias / Selektionsbias)

Das Modell lernt aus **vergangenen Kreditentscheidungen** – darin stecken zwei Quellen systematischer Verzerrung:

1. **Selektionsbias (Reject Inference Problem):** Nur für *bewilligte* Anträge ist das Ausfall-Outcome bekannt. Das Modell lernt nicht „wer generell ausfällt", sondern „wer unter den Bewilligten ausgefallen ist" – eine verzerrte Stichprobe. Abgelehnte Anträge, die möglicherweise kreditwürdig gewesen wären, fehlen.
2. **Bias in, bias out:** Wurden historisch bestimmte Gruppen systematisch benachteiligt (aus diskriminierenden oder strukturellen Gründen), reproduziert und verstärkt das Modell diese Muster.

### 5.3 Proxy-Variablen für sensible Merkmale

Das Modell verwendet keine direkten Diskriminierungsmerkmale in den Top-Features. Dennoch können verschiedene Merkmale als **indirekte Proxys** wirken:

| Merkmal | Potenzielle Proxy-Funktion |
|---------|---------------------------|
| `age_years` / `DAYS_BIRTH` | Direktes Altersmerkmal (geschützt nach AGG § 1) |
| `NAME_EDUCATION_TYPE` | Proxy für sozioökonomische Herkunft, Migrationsgeschichte |
| `NAME_FAMILY_STATUS` | Korreliert mit Geschlecht (z. B. „Alleinerziehend" häufiger weiblich) |
| `REGION_RATING_CLIENT` | Geografischer Proxy – Redlining-Risiko (Benachteiligung ganzer Stadtteile) |
| `AMT_INCOME_TOTAL` | Proxy für sozioökonomische Klasse |

> Dass kein sensibles Merkmal in den Top-5 erscheint, bedeutet **nicht**, dass das Modell fair ist. Indirekte Diskriminierung über Proxys kann rechtlich und ethisch genauso problematisch sein wie direkte Diskriminierung.

### 5.4 Fairness-Definitionen und ihre Unvereinbarkeit

„Fairness" ist kein eindeutiger Begriff. Es gibt mehrere Definitionen, die unter bestimmten Bedingungen **mathematisch unvereinbar** sind (Kleinberg et al., 2016; Chouldechova, 2017):

- **Demographic Parity:** Gleiche Bewilligungsquote für alle Gruppen.
- **Equal Opportunity:** Gleicher Recall (True Positive Rate) für alle Gruppen.
- **Predictive Parity (Calibration):** Gleiche Precision – wenn das Modell „kein Ausfall" sagt, trifft das gleich häufig für alle Gruppen zu.

Diese drei Definitionen können **nicht gleichzeitig** erfüllt sein, wenn die Basisausfallraten zwischen Gruppen unterschiedlich sind (was typischerweise der Fall ist). Eine Entscheidung für eine Fairness-Definition ist daher ein normativ-politischer Akt, kein rein technischer.

### 5.5 Vorhersage ≠ Entscheidung

Das Modell schätzt eine **Wahrscheinlichkeit** – die Kreditentscheidung ist ein separater, wertbehafteter Schritt:
- Welche Schwelle (Threshold) ist akzeptabel? → abhängig von Kosten falscher Ablehnungen vs. tatsächlicher Ausfälle.
- Gibt es Härtefälle, die menschliche Prüfung erfordern?
- Wer trägt die Verantwortung für eine automatisierte Ablehnung?

### 5.6 Regulatorischer Rahmen (EU/DE)

- **DSGVO Art. 22:** Betroffene haben das Recht, nicht ausschließlich einer automatisierten Entscheidung unterworfen zu werden, die erhebliche Auswirkungen auf sie hat. Der genaue Umfang des „Rechts auf Erklärung" (Erwägungsgrund 71) ist juristisch umstritten.
- **EU AI Act (Verordnung 2024/1689):** Kredit-Scoring-Systeme fallen unter **hochrisikante KI-Systeme** (Anhang III). Diese unterliegen strengen Anforderungen an Transparenz, Genauigkeit, Robustheit, menschliche Aufsicht und Dokumentation.
- **AGG (Allgemeines Gleichbehandlungsgesetz):** Verbietet Diskriminierung u. a. nach Alter, Geschlecht, ethnischer Herkunft – auch **mittelbar** über neutrale Kriterien.

### 5.7 Datenschutz

Auch aggregierte oder pseudonymisierte Finanzdaten sind potenziell re-identifizierbar. DSGVO-Grundsätze:
- **Zweckbindung:** Daten dürfen nur für den erhobenen Zweck verwendet werden.
- **Datensparsamkeit:** Nur tatsächlich benötigte Merkmale sollen verarbeitet werden.

---

> **Klare Einordnung:**  
> Dieses Modell ist ein **akademisches Analyseprojekt** auf einem öffentlich verfügbaren Wettbewerbsdatensatz (Kaggle, 2018). Es darf **nicht** als alleinige Grundlage für reale Kreditentscheidungen dienen. Reale Systeme erfordern: regulatorische Konformität (EU AI Act, DSGVO, AGG), ein Fairness-Audit nach Gruppen, menschliche Aufsicht und transparent begründbare Einzelentscheidungen.

## 6. Limitationen dieses Projekts (ehrlich & vollständig)

### Methodische Limitationen

**Kein zeitlicher Split (größte Einschränkung):**  
Ein realistisches Kredit-Scoring-Modell muss zeitlich validiert werden: Training auf älteren Anträgen, Test auf neueren. Der hier verwendete zufällige Split erlaubt „Lernen aus der Zukunft" (Look-ahead Bias) und überschätzt die Generalisierungsfähigkeit auf zukünftige Anträge.

**Selektionsbias / Reject Inference:**  
Nur für bewilligte Anträge ist das Ausfall-Outcome beobachtbar. Das Modell wurde auf einer verzerrten Stichprobe trainiert – die wahre Performance auf dem vollständigen Antragsstrom (inkl. abgelehnter Anträge) wäre schlechter und kaum messbar.

**Aggregations-Trade-off:**  
Die zeitliche Dynamik der Nebentabellen (z. B. Verschlechterung der Zahlungsfähigkeit über die Zeit) geht bei der Aggregation (Mittelwert, letzter Wert, Summe) weitgehend verloren. Sequenzielle Modelle (LSTM, Transformer) könnten diese Information besser nutzen.

**Begrenztes Hyperparameter-Tuning:**  
Aus Effizienzgründen wurde nur ein kleiner Suchraum abgedeckt. Die berichteten Leistungswerte sind als **untere Schranke** zu verstehen, nicht als optimal erreichbar.

**Kein Threshold-Tuning:**  
Die Standardschwelle 0.5 ist für einen stark unausgewogenen Datensatz (≈8 % Positive) suboptimal. Eine produktive Anwendung würde den Schwellenwert nach Kosten-Nutzen-Abwägung (Fehlablehnungskosten vs. Ausfallkosten) optimieren.

### Datenqualitäts-Limitationen

**Unbekannter Fehlmechanismus:** Die Median/Mode-Imputation setzt MCAR voraus – wahrscheinlich eine Vereinfachung, die Ergebnisse leicht verzerrt.

**Datensattalter:** Der öffentliche Home-Credit-Datensatz stammt aus ~2018. Strukturelle Veränderungen (COVID-19, Inflation 2021–2023, Zinswende) sind im Datensatz nicht abgebildet.

**Keine Echtzeitdaten:** In der Praxis fließen aktuelle Kontobewegungen, Marktdaten und Verhaltensmerkmale ein – hier nicht verfügbar.

### Interpretations-Limitationen

**Global vs. lokal:** Permutation Importance und PDPs sind **globale** Maße, gemittelt über alle Datenpunkte. Für die Erklärung einer einzelnen Kreditentscheidung bräuchte man **lokale** Methoden (LIME, SHAP für Einzelinstanzen, Counterfactual Explanations).

**Korrelations-Konfounder:** Alle Wichtigkeitsmaße werden durch Korrelationen zwischen Merkmalen verzerrt. Die wahre kausale Wichtigkeit (via Do-Kalkül oder Instrumentalvariablen) ist aus reinen Beobachtungsdaten nicht zuverlässig ableitbar.

**Kein Fairness-Audit:** Eine gruppenweise Auswertung nach Alter, Geschlecht oder Region fehlt. Ob das Modell bestimmte Gruppen systematisch benachteiligt, wurde nicht untersucht.

---

### Quellen (Auswahl)

- Molnar, C. (2022). *Interpretable Machine Learning* (2nd ed.). [christophm.github.io/interpretable-ml-book]
- Barocas, S., Hardt, M., & Narayanan, A. (2023). *Fairness and Machine Learning.* MIT Press. [fairmlbook.org]
- Kleinberg, J., Mullainathan, S., & Raghavan, M. (2016). Inherent trade-offs in the fair determination of risk scores. *ITCS 2017.*
- Chouldechova, A. (2017). Fair prediction with disparate impact. *Big Data*, 5(2).
- Thomas, L. C. (2009). *Consumer Credit Models.* Oxford University Press.
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions (SHAP). *NeurIPS.*
- EU AI Act – Verordnung (EU) 2024/1689, Anhang III (Hochrisiko-KI-Systeme).
- DSGVO Art. 22 und Erwägungsgrund 71 (Automatisierte Einzelentscheidungen).